In [ ]:
#install.packages("e1071", repos = "https://cloud.r-project.org")  # run once if needed
library(e1071)

#in
train<- read.csv("classification_train.csv", stringsAsFactors = FALSE)
test<- read.csv("classification_test.csv", stringsAsFactors = FALSE)

num_cols <- setdiff(names(train)[sapply(train, is.numeric)], "alwaysAnxious")
X_train <- as.matrix(train[, num_cols, drop = FALSE])
X_test  <- as.matrix(test[, num_cols, drop = FALSE])

center_val <- colMeans(X_train)
scale_val  <- apply(X_train, 2, sd)
scale_val[scale_val == 0] <- 1
X_train_s <- scale(X_train, center = center_val, scale = scale_val)
X_test_s  <- scale(X_test,  center = center_val, scale = scale_val)

class_table <- table(y)
class_weights <- as.numeric(sum(class_table) / (length(class_table) * class_table))
names(class_weights) <- names(class_table)

# model1 0.52

In [20]:
# =========================
# 4.5 用 5-fold CV 估计 macro-F1（插在第4部分和第5部分之间）
# =========================

# (a) 算 macro-F1 的小函数
macro_f1 <- function(truth, pred) {
  f1 <- sapply(levels(truth), function(k) {
    tp <- sum(pred == k & truth == k)
    fp <- sum(pred == k & truth != k)
    fn <- sum(pred != k & truth == k)
    p <- if (tp + fp == 0) 0 else tp / (tp + fp)   # precision
    r <- if (tp + fn == 0) 0 else tp / (tp + fn)   # recall
    if (p + r == 0) 0 else 2 * p * r / (p + r)
  })
  mean(f1)
}

# (b) 把数据分成 5 折（按类别分层，保证每折各类比例一致）
set.seed(1)
fold <- integer(length(y))
for (lev in levels(y)) {
  idx <- sample(which(y == lev))
  fold[idx] <- rep(1:5, length.out = length(idx))
}

# (c) 跑 5 折，每折训练一个 SVM 并在验证折上算 macro-F1
scores <- numeric(5)
for (f in 1:5) {
  tr <- fold != f          # 训练用的行
  va <- fold == f          # 验证用的行
  m <- svm(x = X_train_s[tr, ], y = y[tr],
           kernel = "radial", cost = 2, gamma = 0.05,   # ← 想测别的参数就改这两个数
           class.weights = class_weights, scale = FALSE)
  scores[f] <- macro_f1(y[va], predict(m, X_train_s[va, ]))
}

cat("每折的 macro-F1：", round(scores, 4), "\n")
cat("5 折平均 macro-F1：", round(mean(scores), 4), "\n")

每折的 macro-F1： 0.3493 0.3892 0.4093 0.3856 0.4206 
5 折平均 macro-F1： 0.3908 


In [21]:
# 0.52
set.seed(1)
fin.mod <- svm(
  x = X_train_s, y = y,
  kernel = "radial",
  cost = 2, gamma = 0.05,
  class.weights = class_weights,
  scale = FALSE
)

In [22]:
pred.label <- predict(fin.mod, X_test_s)
pred.label <- as.integer(as.character(pred.label))

In [23]:
write.csv(
  data.frame("RowIndex" = seq_along(pred.label), "Prediction" = pred.label),
  "ClassificationPredictLabel_1.csv",
  row.names = FALSE
)

# model2 0.59


In [25]:
# =========================
# 4.5 用 5-fold CV 估计 macro-F1（插在第4部分和第5部分之间）
# =========================

# (a) 算 macro-F1 的小函数
macro_f1 <- function(truth, pred) {
  f1 <- sapply(levels(truth), function(k) {
    tp <- sum(pred == k & truth == k)
    fp <- sum(pred == k & truth != k)
    fn <- sum(pred != k & truth == k)
    p <- if (tp + fp == 0) 0 else tp / (tp + fp)   # precision
    r <- if (tp + fn == 0) 0 else tp / (tp + fn)   # recall
    if (p + r == 0) 0 else 2 * p * r / (p + r)
  })
  mean(f1)
}

# (b) 把数据分成 5 折（按类别分层，保证每折各类比例一致）
set.seed(2)
fold <- integer(length(y))
for (lev in levels(y)) {
  idx <- sample(which(y == lev))
  fold[idx] <- rep(1:5, length.out = length(idx))
}

# (c) 跑 5 折，每折训练一个 SVM 并在验证折上算 macro-F1
scores <- numeric(5)
for (f in 1:5) {
  tr <- fold != f          # 训练用的行
  va <- fold == f          # 验证用的行
  m <- svm(x = X_train_s[tr, ], y = y[tr],
           kernel = "radial", cost = 1.75, gamma = 0.045,   # ← 想测别的参数就改这两个数
           class.weights = class_weights, scale = FALSE)
  scores[f] <- macro_f1(y[va], predict(m, X_train_s[va, ]))
}

cat("每折的 macro-F1：", round(scores, 4), "\n")
cat("5 折平均 macro-F1：", round(mean(scores), 4), "\n")

每折的 macro-F1： 0.3655 0.4329 0.3882 0.367 0.465 
5 折平均 macro-F1： 0.4037 


In [24]:
# 0.59
# =========================
# 5. Train final weighted RBF SVM
# =========================
set.seed(2)
fin.mod <- svm(
  x = X_train_s,
  y = y,
  kernel = "radial",
  cost = 1.75,
  gamma = 0.045,
  class.weights = class_weights,
  scale = FALSE
)

# =========================
# 6. Predict test labels
# =========================

pred.label <- predict(fin.mod, X_test_s)

# Convert factor labels back to numeric labels
pred.label <- as.integer(as.character(pred.label))


print("Prediction distribution on classification_test:")
print(table(pred.label))

# =========================
# 7. Write Kaggle submission file
# =========================

write.csv(
  data.frame(
    "RowIndex" = seq_along(pred.label),
    "Prediction" = pred.label
  ),
  "ClassificationPredictLabel_2.csv",
  row.names = FALSE
)

[1] "Prediction distribution on classification_test:"
pred.label
-2 -1  0  1  2 
 7 13 36 32  7 
